# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print("Dataset Metadata:")
print("Name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", metadata.identifier)
print("Published Date:", metadata.datePublished)
print("License:", metadata.license)


## 2. Data Overview
Review available record sets, fields, and their IDs.

With the Croissant schema, each data entity (record set, field, column) is referenced by its `@id`. Here we list the dataset's record sets, and for each, the fields and columns available.

In [ ]:
# List all record sets and their fields using @id
# mlcroissant exposes dataset.metadata.record_sets as a list of RecordSet objects

record_sets = dataset.metadata.record_sets  # list of mlc.RecordSet

print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    * Field @id: {field.id} (name: {field.name}, dataType: {field.data_type})")
    print("  Columns:")
    for column in rs.columns:
        print(f"    * Column @id: {column.id} (name: {column.name})")
    print("\n")

# For demonstration, print the first few records from each record set.
for rs in record_sets:
    print(f"First 3 records for RecordSet @id: {rs.id}")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        print(record)
        if i == 2:
            break
    print("\n")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

All record sets and fields must be referenced by their `@id`. Below, data from each record set is loaded into a pandas DataFrame.

In [ ]:
# Extract data from each record set
dataframes = {}
all_record_set_ids = [rs.id for rs in record_sets]
print("Record Set @ids:", all_record_set_ids)

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    df = dataframes[record_set_id]
    print(f"DataFrame for RecordSet @id {record_set_id}:")
    print("Columns (@ids):", df.columns.tolist())
    print("First 5 rows:")
    print(df.head())

# For further analysis, choose the main table record set (assuming the one with most rows is primary)
main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
main_df = dataframes[main_record_set_id]
print(f"Selected main RecordSet for analysis: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

**All references must use the correct `@id` field for columns and group fields as shown above.**

In [ ]:
# Identify numeric fields by @id in the main record set
main_fields = [field for field in [rs for rs in record_sets if rs.id == main_record_set_id][0].fields]
numeric_fields = [field for field in main_fields if field.data_type in ['Integer', 'Float', 'Number']]
print("Numeric Field @ids:")
for field in numeric_fields:
    print(f"- {field.id} (name: {field.name})")

# Pick the first numeric field (or you can choose a specific one)
if numeric_fields:
    numeric_field_id = numeric_fields[0].id
    threshold = main_df[numeric_field_id].mean()
    print(f"Applying threshold (mean): {threshold} for field {numeric_field_id}")
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Try grouping by a categorical field
    group_fields = [field for field in main_fields if field.data_type == 'Text']
    if group_fields:
        group_field_id = group_fields[0].id
        print(f"Grouping by {group_field_id}")
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print("Grouped averages:")
            print(grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example distribution plot of the chosen numeric field and a bar plot showing mean values by a categorical group field.

In [ ]:
if numeric_fields:
    numeric_field_id = numeric_fields[0].id
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    main_df[numeric_field_id].hist(bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Bar plot of mean by group (categorical)
    if group_fields and group_field_id in main_df.columns:
        group_df = main_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10,5))
        plt.bar(group_df[group_field_id], group_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the FAIR^2 dataset using the Croissant schema and `mlcroissant`.
- Using `@id` references for entities, we explored the available record sets, fields, and columns.
- Data extraction and EDA steps showcased methods to filter, normalize, and group data.
- Visualizations highlighted variable distributions and group differences.
- The dataset supports clinical and molecular stratification analysis of second primary colorectal cancer survivors.

For further insights, refer to the mlcroissant documentation and FAIR^2 dataset schema.